# 01 – Loading, checking and cleaning the ambulance data

**Project question:** Does where you live in England affect how fast an ambulance arrives?

**This notebook:**
1. Loads the NHS England Ambulance Quality Indicators data (AmbSYS)
2. Checks what is in it and how good the data is
3. Picks out the response-time measures we need and checks we understand them
4. Fixes problems and saves a clean, tidy file for the next notebooks
5. Makes two first charts

**Data source:** NHS England, *Ambulance Quality Indicators* – AmbSYS CSV (Sept 2018 – Sept 2026, published 8 Oct 2026).
This is public, aggregated data: no patient-level or identifiable information is used.

## 1. Setup

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 30)
plt.rcParams["figure.figsize"] = (10, 5)

## 2. Load the raw data

Folder layout this notebook expects:

```
ambulance-project/
├── data/
│   ├── raw/          <- put the AmbSYS CSV here
│   └── processed/    <- this notebook saves the clean file here
├── notebooks/        <- this notebook lives here
└── outputs/figures/  <- charts are saved here
```

We look for any CSV in `data/raw` whose name starts with `AmbSYS`, so the exact file name doesn't matter.

In [ ]:
PROJECT_DIR = Path("..")
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
FIG_DIR = PROJECT_DIR / "outputs" / "figures"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

raw_files = sorted(RAW_DIR.glob("AmbSYS*.csv"))
assert raw_files, f"No AmbSYS CSV found in {RAW_DIR.resolve()}"
raw_path = raw_files[-1]
print("Loading:", raw_path.name)

raw = pd.read_csv(raw_path, low_memory=False)
print("Rows, columns:", raw.shape)
raw.iloc[:5, :12]

## 3. What is in the file?

Each row is **one organisation in one month**. There are three kinds of organisation mixed together:
- **England** (the national total)
- **NHS regions** (7 of them)
- **Ambulance trusts** (the services that actually send ambulances)

We can tell them apart using the `Region` column: trusts have a region code like `Y56`.

In [ ]:
orgs = raw[["Region", "Org Code", "Org Name"]].drop_duplicates()
orgs

**Data quality issue spotted:** some organisations appear with more than one name over time.
That's why we will always use the `Org Code`, not the name, to identify an organisation.

In [ ]:
names_per_code = orgs.groupby("Org Code")["Org Name"].nunique()
names_per_code[names_per_code > 1]

In [ ]:
orgs[orgs["Org Code"].isin(names_per_code[names_per_code > 1].index)]

## 4. Missing values

All the measure columns were read as text. That's because NHS England uses a full stop (`.`) to mean
"not available". We'll count these, then turn them into proper missing values (`NaN`).

In [ ]:
measure_cols = [c for c in raw.columns if c.startswith("A")]
dot_counts = (raw[measure_cols] == ".").sum().sort_values(ascending=False)
dot_counts[dot_counts > 0].head(15)

In [ ]:
df = pd.concat(
    [raw.drop(columns=measure_cols), raw[measure_cols].apply(pd.to_numeric, errors="coerce")],
    axis=1,
)
df["date"] = pd.to_datetime(dict(year=df["Year"], month=df["Month"], day=1))
print("Date range:", df["date"].min().date(), "to", df["date"].max().date())

## 5. Picking the measures we need

The columns have codes (A8, A25, ...) instead of names. The official meanings are in NHS England's
*AmbSYS indicator list*. The ones we need are:

| Category | What it means | Calls (count) | Total time (seconds) | Mean (seconds) | 90th centile (seconds) | National mean target |
|---|---|---|---|---|---|---|
| **C1** | Life-threatening (e.g. cardiac arrest) | A8 | A24 | A25 | A26 | 7 min |
| **C2** | Emergency (e.g. stroke, heart attack) | A10 | A30 | A31 | A32 | 18 min |
| **C3** | Urgent | A11 | A33 | A34 | A35 | – (90th centile: 2 hrs) |
| **C4** | Less urgent | A12 | A36 | A37 | A38 | – (90th centile: 3 hrs) |

**Checking we've got this right:** if the codes mean what we think, then
*total time ÷ number of calls* should equal the *mean*. Let's test it on every row.

In [ ]:
checks = {
    "C1": ("A24", "A8", "A25"),
    "C2": ("A30", "A10", "A31"),
    "C3": ("A33", "A11", "A34"),
    "C4": ("A36", "A12", "A37"),
}
for cat, (total, count, mean) in checks.items():
    calc = df[total] / df[count]
    diff = (calc - df[mean]).abs()
    print(f"{cat}: biggest difference = {diff.max():.2f} seconds")

The differences are tiny (a few seconds at most – just rounding), so our reading of the codes is correct. ✅

## 6. Build a clean, tidy table

We keep only what we need, give the columns readable names, and label each row as England, region or trust.
We also give each trust one short, consistent name.

In [ ]:
TRUST_NAMES = {
    "R1F": "Isle of Wight",
    "RRU": "London",
    "RX6": "North East",
    "RX7": "North West",
    "RX8": "Yorkshire",
    "RX9": "East Midlands",
    "RYA": "West Midlands",
    "RYC": "East of England",
    "RYD": "South East Coast",
    "RYE": "South Central",
    "RYF": "South Western",
}

def org_level(row):
    if row["Org Code"] == "Eng":
        return "England"
    if row["Org Code"] in TRUST_NAMES:
        return "Trust"
    return "Region"

df["level"] = df.apply(org_level, axis=1)
df["name"] = df["Org Code"].map(TRUST_NAMES).fillna(df["Org Name"])
df.groupby("level")["Org Code"].nunique()

In [ ]:
COLUMNS = {
    "A8": "c1_calls",  "A24": "c1_total_sec", "A25": "c1_mean_sec", "A26": "c1_p90_sec",
    "A10": "c2_calls", "A30": "c2_total_sec", "A31": "c2_mean_sec", "A32": "c2_p90_sec",
    "A11": "c3_calls", "A33": "c3_total_sec", "A34": "c3_mean_sec", "A35": "c3_p90_sec",
    "A12": "c4_calls", "A36": "c4_total_sec", "A37": "c4_mean_sec", "A38": "c4_p90_sec",
}

clean = (
    df[["date", "level", "Org Code", "name"] + list(COLUMNS)]
    .rename(columns={"Org Code": "org_code", **COLUMNS})
    .sort_values(["level", "name", "date"])
    .reset_index(drop=True)
)

# Minutes are easier to read than seconds
for cat in ["c1", "c2", "c3", "c4"]:
    clean[f"{cat}_mean_min"] = clean[f"{cat}_mean_sec"] / 60
    clean[f"{cat}_p90_min"] = clean[f"{cat}_p90_sec"] / 60

clean.head()

### Check: does every organisation have every month?
(The file has a row for every month, but some rows are blank, so we count rows that actually have data.)
If some months are missing, comparisons could be unfair.

In [ ]:
# Count only months where Category 2 data is actually filled in
has_data = clean.dropna(subset=["c2_calls"])
months_per_org = has_data.groupby(["level", "name"])["date"].agg(["min", "max", "count"])
months_per_org

**Note:** in the first months (late 2017) not every trust sent data yet, so 2017 England totals are incomplete.
We'll leave 2017 out when comparing years.

In [ ]:
# Any duplicate organisation-month rows?
dupes = clean.duplicated(["org_code", "date"]).sum()
print("Duplicate org-month rows:", dupes)

## 7. Save the clean file

In [ ]:
out_path = PROCESSED_DIR / "ambulance_response_times_clean.csv"
clean.to_csv(out_path, index=False)
print("Saved", len(clean), "rows to", out_path)

## 8. First look: Category 2 response times in England

Category 2 covers emergencies like strokes and heart attacks. The national target is a **mean of 18 minutes**.

In [ ]:
eng = clean[clean["level"] == "England"]

fig, ax = plt.subplots()
ax.plot(eng["date"], eng["c2_mean_min"], label="Category 2 mean")
ax.axhline(18, color="red", linestyle="--", label="18-minute target")
ax.set_title("England: average Category 2 ambulance response time")
ax.set_ylabel("Minutes")
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "01_england_c2_trend.png", dpi=150)
plt.show()

In [ ]:
# Yearly average, weighted by number of calls (see note below)
yearly = (
    eng[eng["date"].dt.year >= 2018]
       .assign(year=lambda x: x["date"].dt.year)
       .groupby("year")[["c2_total_sec", "c2_calls"]].sum()
)
yearly["c2_mean_min"] = (yearly["c2_total_sec"] / yearly["c2_calls"] / 60).round(1)
yearly[["c2_calls", "c2_mean_min"]]

## 9. First look: differences between ambulance trusts (last 12 months)

**Important method point:** to get a 12-month average we should *not* just average the 12 monthly means,
because busy months have more patients. Instead we add up all the waiting time and divide by all the calls.
This is called a **weighted average**.

In [ ]:
latest = clean["date"].max()
last_12 = clean[(clean["level"] == "Trust") & (clean["date"] > latest - pd.DateOffset(months=12))]

trust_summary = last_12.groupby("name")[["c1_total_sec", "c1_calls", "c2_total_sec", "c2_calls"]].sum()
trust_summary["c1_mean_min"] = trust_summary["c1_total_sec"] / trust_summary["c1_calls"] / 60
trust_summary["c2_mean_min"] = trust_summary["c2_total_sec"] / trust_summary["c2_calls"] / 60
trust_summary = trust_summary.sort_values("c2_mean_min")
trust_summary[["c2_calls", "c1_mean_min", "c2_mean_min"]].round(1)

In [ ]:
fig, ax = plt.subplots()
ax.barh(trust_summary.index, trust_summary["c2_mean_min"])
ax.axvline(18, color="red", linestyle="--", label="18-minute target")
ax.set_title(f"Average Category 2 response by ambulance trust\n(12 months to {latest:%B %Y})")
ax.set_xlabel("Minutes")
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "01_trust_c2_last12.png", dpi=150)
plt.show()

## 10. Summary

**Data checks**
- Data runs monthly from the start of the file to September 2026 for England, 7 regions and 11 ambulance trusts.
- Missing values were coded as `.` and have been converted to proper missing values.
- One trust (West Midlands) appears under two names, so organisations are matched by code, not name.
- The meaning of each response-time code was confirmed by recalculating the means from totals and counts.

**First findings**
- Category 2 response times rose sharply after 2020, peaked in 2022, and are still well above the 18-minute target.
- There is a big gap between trusts: the slowest trust's average is almost double the fastest.

**Next notebook (02):** add area data (deprivation and how rural each area is) to explore *why* some areas wait longer.